# Exercise 12: Resonance detective

**Block 4: Resonances**

**Learning objectives**
* Identify an unknown magnet and the resonance it drives, only from phase-space pictures and action plots.
* Practise the "reading rules" of Exercise 11: number of islands <-> order of the resonance, which plane is
  affected <-> normal or skew, exchange vs growth <-> difference vs sum resonance.

**Motivation.** In a real machine nobody tells you which magnet is causing trouble. You see the beam
(turn-by-turn data from beam position monitors, losses, tune spectra) and must work out what is
going on. Here six mystery machines are waiting for you.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import lattices as L, tracking as T, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()
DATA = _root / "notebooks" / "data" / "resonance_detective.npz"

## The evidence

For each mystery machine you get three plots, all in normalised coordinates:

* **left:** Poincaré section in $x$: 12 particles started at increasing $\hat{x}$ (and a tiny $\hat{y}$);
* **middle:** the same in $y$: particles started at increasing $\hat{y}$ (and a tiny $\hat{x}$);
* **right:** the actions $J_x$ (blue) and $J_y$ (red) of one particle started mostly in $x$, turn by turn
  (log scale).

You also know the machine tunes ($Q_x$, $Q_y$): in a real machine they are measured all the time.
Particles that get lost simply stop appearing.

The suspects: normal or skew quadrupole, sextupole or octupole (one magnet per machine).

In [ ]:
data = np.load(DATA)
n_mysteries = len({k.split("_")[0] for k in data.files})

for k in range(1, n_mysteries + 1):
    d = {key.split("_", 1)[1]: data[key] for key in data.files if key.startswith(f"m{k}_")}
    lim = 1.6 * float(d["amax"]) * MM
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, plane in zip(axs[:2], ["x", "y"]):
        u, pu = d[f"{plane}_u"], d[f"{plane}_pu"]
        ax.scatter(u * MM, pu * MM, s=0.3, c=np.broadcast_to(np.arange(u.shape[1]), u.shape), cmap="viridis")
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        P.phase_space_labels(ax, plane, True)
        ax.set_title(f"{plane} plane")
    axs[2].semilogy(d["Jx"] * 1e6, color=COLORS["x"], lw=1, label="Jx")
    axs[2].semilogy(d["Jy"] * 1e6, color=COLORS["y"], lw=1, label="Jy")
    axs[2].set_xlabel("turn"); axs[2].set_ylabel("action [mm mrad]"); axs[2].legend()
    fig.suptitle(f"Mystery machine {k}:  Qx = {float(d['Qx']):.3f},  Qy = {float(d['Qy']):.3f}", fontsize=13)
    plt.show()

## Your verdict

**Predict first / think.** For each machine, write down which plane is affected, whether you see islands
(how many?), runaway growth, or exchange between the planes. Then use the tunes: which combination
$m Q_x + n Q_y$ is close to an integer?

Fill in the table (double-click to edit):

| # | magnet (normal/skew, quad/sext/oct) | resonance (e.g. $3Q_x = 1$) | your evidence |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |
| 6 | | | |

Hints:
* The order of the resonance is the number of islands (or corners of the separatrix).
* Normal multipoles leave $y = 0$ invariant. Which skew multipoles leave $x = 0$ invariant?
* A skew quadrupole couples the planes: the difference resonance exchanges amplitude, the sum resonance
  makes both grow.

## Reflection questions

1. Which single piece of evidence was most useful to tell a skew from a normal magnet?
2. Machine 6 has bounded motion despite being near a resonance, machine 1 does not. Why?
3. In a real machine you cannot see the whole phase space, only $x$ (and maybe $p_x$) of the beam centroid
   turn by turn. Which of these signatures could you still detect?